# Предсказание пола по транзакциям — Sber 2026 DS

**Цель:** ROC AUC > 0.88 и Accuracy > 0.80

**Решение:** `solution_super_200.py` — super-ensemble: все типы фич (685 кандидатов) → top-200 → 5-fold LGB.

## 1. Установка зависимостей

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
print("Dependencies installed.")

## 2. Загрузка данных

Скачивает `train.csv`, `val.csv`, `test.csv` из HuggingFace в `./data/`.

In [ ]:
import os
import download_dataset

os.makedirs("data", exist_ok=True)

if all(os.path.exists(f"data/{s}.csv") for s in ["train", "val", "test"]):
    print("Data already downloaded.")
else:
    download_dataset  # module runs on import; re-import triggers re-run
    print("Data downloaded.")

for split in ["train", "val", "test"]:
    size = os.path.getsize(f"data/{split}.csv") // 1024 // 1024
    print(f"  data/{split}.csv  {size} MB")

## 3. Вычисление gender-embeddings (spaCy)

Считает косинусную близость описаний MCC/TR к мужским/женским словам.  
Сохраняет `results/gender_embeddings/mcc_gender.csv` и `tr_gender.csv`.

In [ ]:
import importlib
import pandas as pd
import compute_gender_embeddings

if (os.path.exists("results/gender_embeddings/mcc_gender.csv") and
    os.path.exists("results/gender_embeddings/tr_gender.csv")):
    print("Gender embeddings already computed.")
else:
    import spacy.cli
    spacy.cli.download("ru_core_news_md")

    importlib.reload(compute_gender_embeddings)
    compute_gender_embeddings.main()

mcc_emb = pd.read_csv("results/gender_embeddings/mcc_gender.csv")
tr_emb  = pd.read_csv("results/gender_embeddings/tr_gender.csv")
print(f"MCC embeddings: {len(mcc_emb)} codes")
print(f"TR  embeddings: {len(tr_emb)} types")
display(mcc_emb.head())

## 4. Обучение модели

**Super-200:** все типы фич (685 кандидатов) → top-200 по LGB importance → 5-fold ensemble.

In [ ]:
import importlib
import solution_super_200
importlib.reload(solution_super_200)

metrics = solution_super_200.main()

## 5. Итоговые метрики

In [ ]:
auc = metrics["auc"]
acc = metrics["accuracy"]

print("=" * 40)
print(f"  Model:     super_200  (200 features)")
print(f"  ROC AUC:   {auc}  {'✓' if auc > 0.88 else '✗'} (target > 0.88)")
print(f"  Accuracy:  {acc}  {'✓' if acc > 0.80 else '✗'} (target > 0.80)")
print(f"  Precision: {metrics['precision']}")
print(f"  Recall:    {metrics['recall']}")
print(f"  Train:     {metrics['train_time']}s")
print("=" * 40)

## 6. Сравнение всех экспериментов

In [ ]:
import json
import pandas as pd

with open("results/overall.json") as f:
    overall = json.load(f)

rows = []
for e in overall:
    m = e["metrics"]
    rows.append({
        "branch": e["branch"].replace("solution_2026_04_15_", ""),
        "AUC": m["auc_score"],
        "Accuracy": m["accuracy_score"],
        "n_features": e.get("n_features", ""),
        "train_s": e.get("timing", {}).get("train_time_s", ""),
    })

df_res = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
df_res["AUC ✓"] = df_res["AUC"].apply(lambda x: "✓" if x > 0.88 else "")
df_res["Acc ✓"] = df_res["Accuracy"].apply(lambda x: "✓" if x > 0.80 else "")
display(df_res)